## Create Next Batch

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
control_table=f"{catalog_name}.{control_schema}.batch_control"

In [0]:
# from pyspark.sql import Row
# from pyspark.sql import  functions as F
# if v_batch_id:
#     in_progress_df=(
#         spark.createDataFrame(
#             [Row(batch_id=v_batch_id,status="in_progress")]
#         )
#         .withColumn("created_timestamp",F.current_timestamp())
#         .withColumn("updated_timestamp",F.current_timestamp())
#     )
#     (
#         in_progress_df.write
#         .format("delta")
#         .mode("append")
#         .saveAsTable(control_table)
#     )
#     print(f"Marked batch {v_batch_id} as in progress")
# else:
#     raise Exception("batch _id is missing")


In [0]:
from pyspark.sql import functions as F

if not v_batch_id:
    raise Exception("batch_id is missing")

spark.sql(f"""
MERGE INTO {control_table} t
USING (SELECT '{v_batch_id}' AS batch_id) s
ON t.batch_id = s.batch_id
WHEN MATCHED AND t.status = 'failed' THEN
  UPDATE SET t.status = 'in_progress', t.updated_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (batch_id, status, created_timestamp, updated_timestamp)
  VALUES (s.batch_id, 'in_progress', current_timestamp(), current_timestamp())
""")

print(f"Marked batch {v_batch_id} as in_progress")